## Import essential python packages

In [ ]:
# import spacec first
import spacec as sp
import glob
import numpy as np
import pandas as pd

#import standard packages
import os
import warnings
import matplotlib.pyplot as plt
import scanpy as sc

import anndata as ad
warnings.filterwarnings('ignore')

import fcsparser

from collections import Counter
import matplotlib.backends.backend_pdf

import networkx as nx
from sklearn.neighbors import kneighbors_graph
plt.rc('axes', grid=False)  # remove gridlines
sc.settings.set_figure_params(dpi=80, facecolor='white') # set dpi and background color for scanpy figures


from scipy.stats import entropy, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import seaborn as sns
from statannotations.Annotator import Annotator

from scipy.spatial.distance import pdist
from statsmodels.stats.multitest import multipletests
from scipy.stats import mannwhitneyu, wilcoxon


from math import factorial
from itertools import permutations
from scipy.stats import spearmanr, rankdata

from scipy.spatial import Delaunay

from scipy.stats import rankdata


## Initialize variables

In [ ]:
# Specify the path to the data
root_path = "~/CODEX_08262025"
data_filename = root_path + "/celldata.xlsx" # where the data is stored

# where you want to store the output
output_dir = root_path + '/Analysis/Requested_Mar17_Refined/' # inset your own path
os.makedirs(output_dir, exist_ok=True)

## Load Ann data

In [ ]:
adata = sc.read_h5ad("%s/Analysis/CN_Corrected_Dormant/adata_annotated_cn.h5ad" % root_path)
adata

In [ ]:
adata.obs['unique_region'] = [uniq_reg.rstrip("\n\r").split("_PD_")[0] for uniq_reg in adata.obs['unique_region']]

### Review metadata

In [ ]:
o = adata.obs
print(o['celltypes'].value_counts())
print(o['CN_k10_n7_annot'].value_counts())
print(o['condition_final'].value_counts())
print(o.groupby('condition_final')['code'].nunique())
print(o.groupby(['code','condition_final'])['unique_region'].nunique())

### Spatial adjacency

In [ ]:
## Core-level adjacency

## Init values
CT, ZONE, NPERM, SEED = 'M1-like Active', 'Vascular Zone', 300, 0
rng = np.random.default_rng(SEED)

def delaunay_edges(xy):
    tri = Delaunay(xy)
    s = tri.simplices
    e = np.vstack([s[:, [0, 1]], s[:, [1, 2]], s[:, [0, 2]]])
    e.sort(axis=1)
    return np.unique(e, axis=0)

rows = []
for reg, g in adata.obs.groupby('unique_region', observed=True):
    g  = g.reset_index(drop=True)  
    xy = g[['X', 'Y']].to_numpy(float)  ## Extract x-y coodinates
    in_zone = (g['CN_k10_n7_annot'] == ZONE).to_numpy()  ## Is cell in Zone?
    is_m1   = ((g['celltypes'] == CT) & ~in_zone).to_numpy()   # overlap excluded

    rec = dict(unique_region=reg, code=g['code'].iloc[0],
               condition_final=g['condition_final'].iloc[0], TMA=g['TMA'].iloc[0],
               n_cells_core=len(g), n_zone=int(in_zone.sum()),
               n_m1_adj=int(is_m1.sum()),
               n_m1_total=int((g['celltypes'] == CT).sum()))
    
    ## is total cells less than 4 or no cells in the zone or no m1 cells adjacency of zone?
    if len(g) < 4 or rec['n_zone'] == 0 or rec['n_m1_adj'] == 0:
        rec['status'] = ('too_few_cells' if len(g) < 4
                         else 'no_zone' if rec['n_zone'] == 0 else 'no_m1')
        rows.append(rec); continue

    ed = delaunay_edges(xy)
    u, v = ed[:, 0], ed[:, 1]
    cross = lambda m: int(((m[u] & in_zone[v]) | (in_zone[u] & m[v])).sum())

    obs  = cross(is_m1)
    pool = np.flatnonzero(~in_zone)                 # M1 labels move among non-zone cells
    null = np.empty(NPERM)
    for i in range(NPERM):
        m = np.zeros(len(g), bool)
        m[rng.choice(pool, rec['n_m1_adj'], replace=False)] = True
        null[i] = cross(m)

    mu, sd = null.mean(), null.std(ddof=1)
    rec.update(status='ok', n_edges=len(ed), obs_contacts=obs,
               exp_mean=mu, exp_sd=sd,
               log2_enrichment=np.log2((obs + .5) / (mu + .5)),
               z_score=(obs - mu) / sd if sd > 0 else np.nan)
    rows.append(rec)
    

core = pd.DataFrame(rows)
print(pd.crosstab(core.condition_final, core.status))
core

In [ ]:
# patient-level 

ok   = core[core.status == 'ok']
keep = ok[ok.exp_mean >= 5]          # QC: expected contacts too low to measure

def agg(g):
    return pd.Series({
        'n_cores_evaluable': len(g),
        'obs_sum':  g.obs_contacts.sum(),
        'exp_sum':  g.exp_mean.sum(),
        'mean_z':          g.z_score.mean(),
        'weighted_mean_z': np.average(g.z_score, weights=g.exp_mean),
        'mean_log2':       g.log2_enrichment.mean(),
        'pooled_log2':     np.log2((g.obs_contacts.sum() + .5) / (g.exp_mean.sum() + .5)),
    })

## all set
pt = ok.groupby(['code', 'condition_final'], observed=True).apply(agg).reset_index()
pt['n_cores_total'] = (core.groupby(['code', 'condition_final'], observed=True)
                           .size()
                           .reindex(pd.MultiIndex.from_frame(
                               pt[['code', 'condition_final']])).values)

## after QC
pt2 = keep.groupby(['code', 'condition_final'], observed=True).apply(agg).reset_index()
pt2['n_cores_total'] = (core.groupby(['code', 'condition_final'], observed=True)
                            .size()
                            .reindex(pd.MultiIndex.from_frame(
                                pt2[['code', 'condition_final']])).values)

print(len(pt), "patients")
print(pt.groupby('condition_final').size())

print(len(pt2), 'patients')
print(pt2.groupby('condition_final').size())


## Print out the tables to Excel files
with pd.ExcelWriter(f'{output_dir}/VascularZone_M1_adjacency.xlsx') as w:
    core.to_excel(w, sheet_name='per_core',         index=False)
    pt.to_excel(w,   sheet_name='per_patient_all',  index=False)
    pt2.to_excel(w,  sheet_name='per_patient_QC',   index=False)

In [ ]:
## Box plot and pairwise analysis

# ---------------- config ----------------
y_col       = "mean_z"
plot_title  = "Vascular Zone - M1 Active adjacency by Patient Condition"
y_label     = "Mean across cores of the per-core z-score"
order       = ['Uninfected', 'Dormant', 'Infected']
palette     = {'Uninfected': '#619cff', 'Dormant': '#b4c640', 'Infected': '#f8766d'}
SHOW_BRACKETS = True          # pairwise FDR brackets
SHOW_ONESAMP  = True          # per-group Wilcoxon vs 0
# ----------------------------------------

df = pt2.dropna(subset=[y_col]).copy()

print(f'n per group:\n{df.groupby("condition_final").size()}\n')

# ---- pairwise Mann-Whitney + BH ----
pairs = [("Uninfected", "Infected"), ("Uninfected", "Dormant"), ("Infected", "Dormant")]
raw_p = []
for p1, p2 in pairs:
    g1 = df.loc[df.condition_final == p1, y_col]
    g2 = df.loc[df.condition_final == p2, y_col]
    raw_p.append(mannwhitneyu(g1, g2).pvalue)

p_adj = multipletests(raw_p, method='fdr_bh')[1]

print('pairwise (Mann-Whitney, BH-adjusted):')
for (p1, p2), r, a in zip(pairs, raw_p, p_adj):
    print(f'  {p1:12s} vs {p2:12s}  raw={r:.4f}  adj={a:.4f}')

def stars(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else "ns"

annotations = [f"p = {p:.3f} ({stars(p)})" if p >= .001 else f"p < 0.001 ({stars(p)})"
               for p in p_adj]

# ---- one-sample Wilcoxon vs 0 ----
onesamp = {}
for g, d in df.groupby('condition_final'):
    v = d[y_col]
    onesamp[g] = (len(v), v.median(), wilcoxon(v).pvalue if len(v) >= 5 else np.nan)

print('\none-sample (Wilcoxon vs 0):')
for g in order:
    n, med, p = onesamp[g]
    print(f'  {g:12s} n={n:2d}  median={med:+.3f}  p={p:.4f}')


    
# ---- group descriptives ----
desc = (df.groupby('condition_final')[y_col]
          .agg(n='size', median='median',
               q1=lambda x: x.quantile(.25), q3=lambda x: x.quantile(.75),
               min='min', max='max')
          .reindex(order).reset_index())

# ---- one-sample Wilcoxon vs 0, with matched-pairs rank-biserial ----
rows = []
for g in order:
    v = df.loc[df.condition_final == g, y_col].dropna().values
    nz = v[v != 0]
    r = rankdata(np.abs(nz))
    wpos, wneg = r[nz > 0].sum(), r[nz < 0].sum()
    rb = (wpos - wneg) / (wpos + wneg) if (wpos + wneg) else np.nan
    res = wilcoxon(v) if len(v) >= 5 else None
    rows.append({
        'group': g, 'n': len(v), 'median': v.mean() * 0 + np.median(v),
        'statistic_W': res.statistic if res else np.nan,
        'p_value': res.pvalue if res else np.nan,
        'rank_biserial': rb,
        'n_below_zero': int((v < 0).sum()),
    })
onesamp_tbl = pd.DataFrame(rows)

# ---- pairwise Mann-Whitney, raw + BH, with rank-biserial ----
rows = []
for (p1, p2), raw, adj in zip(pairs, raw_p, p_adj):
    a = df.loc[df.condition_final == p1, y_col].dropna().values
    b = df.loc[df.condition_final == p2, y_col].dropna().values
    U = mannwhitneyu(a, b).statistic
    rows.append({
        'group_1': p1, 'n_1': len(a), 'median_1': np.median(a),
        'group_2': p2, 'n_2': len(b), 'median_2': np.median(b),
        'statistic_U': U,
        'p_raw': raw, 'p_BH': adj,
        'rank_biserial': 2 * U / (len(a) * len(b)) - 1,
    })
pairwise_tbl = pd.DataFrame(rows)

print(desc.round(4).to_string(index=False), '\n')
print(onesamp_tbl.round(4).to_string(index=False), '\n')
print(pairwise_tbl.round(4).to_string(index=False))

# ---- write out ----
with pd.ExcelWriter(f'{output_dir}/VascularZone_M1_adjacency_stats.xlsx') as w:
    desc.to_excel(w,          sheet_name='group_descriptives', index=False)
    onesamp_tbl.to_excel(w,   sheet_name='onesample_vs_zero',  index=False)
    pairwise_tbl.to_excel(w,  sheet_name='pairwise_BH',        index=False)
    
    
    
# ---- plot ----
plt.figure(figsize=(6, 8))
sns.set_theme(style="whitegrid")

ax = sns.boxplot(data=df, x='condition_final', y=y_col, order=order,
                 hue='condition_final', palette=palette, legend=False,
                 showfliers=False, width=0.5, boxprops={'alpha': 0.7})

sns.stripplot(data=df, x='condition_final', y=y_col, order=order,
              color='black', alpha=0.6, jitter=True, size=6)

ax.axhline(0, ls='--', lw=1, color='grey', zorder=0)
#ax.text(ax.get_xlim()[1], 0, ' random\n mixing', va='center', ha='left',
#        fontsize=9, color='grey')

if SHOW_BRACKETS:
    annotator = Annotator(ax, pairs, data=df, x='condition_final', y=y_col, order=order)
    annotator.set_custom_annotations(annotations)
    annotator.configure(loc='inside', fontsize=10)
    annotator.annotate()

if SHOW_ONESAMP:
    lo = df[y_col].min()
    for i, g in enumerate(order):
        n, med, p = onesamp[g]
        lab = 'n.d.' if np.isnan(p) else (f'p = {p:.4f}')
        ax.text(i, lo - 0.45, f'{lab}\nn = {n}', ha='center', va='top', fontsize=9)
    ax.set_ylim(lo - 1.4, ax.get_ylim()[1])

plt.title(plot_title, fontsize=14, pad=15)
plt.ylabel(y_label, fontsize=12)
plt.xlabel('')
plt.tight_layout()

plt.savefig(f'{output_dir}/VascularZone_M1_adjacency_Boxplot', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

### Count matrix (Regions X celltypes)

In [ ]:
cnt_df = adata.obs.groupby(['unique_region', 'celltypes']).size().unstack(fill_value=0)
cnt_df

### Vascular Immune Exclusion Ratio (VIER)  

In [ ]:
counts_df = pd.crosstab(adata.obs['unique_region'], adata.obs['CN_k10_n7_annot'])
counts_df.columns

meta_map = adata.obs[['unique_region', 'condition_final', 'code']].drop_duplicates()
df_merged = counts_df.merge(meta_map, on='unique_region')
df_merged

In [ ]:
numerator_cols = [
    'Immune Priming Zone', 
    'Inflammatory Zone'
]
# 'T (pan)'

denominator_cols = [
    'Vascular Zone'
]

df_merged[df_merged['code'] == 'PD_03']

In [ ]:
score_name = "Vascular_Immune_Exclusion_Ratio"

numerator_cols = [
    'Immune Priming Zone', 
    'Inflammatory Zone'
]
# 'T (pan)'

denominator_cols = [
    'Vascular Zone'
]

# The + 1 in the denominator prevents division by zero (Laplace smoothing)
df_merged[score_name] = (
    df_merged[numerator_cols].sum(axis=1) / 
    (df_merged[denominator_cols].sum(axis=1) + 1)
)
# Find the baseline mean of the Raw VIER specifically for Group A
baseline_vier = df_merged[df_merged['condition_final'] == 'Uninfected'][score_name].mean()
#Divide everyone's Raw VIER by the baseline
df_merged['Normalized_VIER'] = df_merged[score_name] / baseline_vier

final_df = df_merged[['unique_region', 'condition_final', 'code', score_name, 'Normalized_VIER']]
final_df

### Antigen Presentation Cell Engagement Ratio (APER)

In [ ]:
counts_df = pd.crosstab(adata.obs['unique_region'], adata.obs['celltypes'])
counts_df.columns

meta_map = adata.obs[['unique_region', 'condition_final']].drop_duplicates()
df_merged = counts_df.merge(meta_map, on='unique_region')

In [ ]:
score_name = "Antigen_Presentation_Cell_Engagement_Ratio"
numerator_cols = [
    'M1-like Active', 
    'DCs Active'
]
# 'T (pan)'

denominator_cols = [
    'M2_TAM Active', 
    'M2_TAM Inactive', 
    'M2_TAM Suppressive', 
    'M1-like Inactive'
]
#    'B Inactive'

# Calculate the Immune Engagement Ratio (IER)
# The + 1 in the denominator prevents division by zero (Laplace smoothing)
df_merged[score_name] = (
    df_merged[numerator_cols].sum(axis=1) / 
    (df_merged[denominator_cols].sum(axis=1) + 1)
)

#print(df_merged)

y_col = score_name
#for col in numerator_cols:
#    y_cols.append(col)
    
#for col in denominator_cols:
#    y_cols.append(col)
    
#y_cols

if not score_name in final_df.columns.values:
    final_df = final_df.merge(df_merged[['unique_region', score_name]], on='unique_region')

    
print(final_df)


### Immune Engagement Ratio (ALL)

In [ ]:
counts_df = pd.crosstab(adata.obs['unique_region'], adata.obs['celltypes'])
counts_df.columns

meta_map = adata.obs[['unique_region', 'condition_final']].drop_duplicates()
df_merged = counts_df.merge(meta_map, on='unique_region')

In [ ]:
score_name = "Immune_Engagement_Ratio"

numerator_cols = [
    'M1-like Active', 
    'DCs Active', 
    'T CD4 NK-like'
]
# 'T (pan)'

denominator_cols = [
    'M2_TAM Active', 
    'M2_TAM Inactive', 
    'M2_TAM Suppressive', 
    'M1-like Inactive'
]
#    'B Inactive'

# Calculate the Immune Engagement Ratio (IER)
# The + 1 in the denominator prevents division by zero (Laplace smoothing)
df_merged['Immune_Engagement_Ratio'] = (
    df_merged[numerator_cols].sum(axis=1) / 
    (df_merged[denominator_cols].sum(axis=1) + 1)
)

if not score_name in final_df.columns.values:
    final_df = final_df.merge(df_merged[['unique_region', score_name]], on='unique_region')

    
print(final_df)

### Active Immune Engagement Ratio

In [ ]:
counts_df = pd.crosstab(adata.obs['unique_region'], adata.obs['celltypes'])
counts_df.columns

meta_map = adata.obs[['unique_region', 'condition_final']].drop_duplicates()
df_merged = counts_df.merge(meta_map, on='unique_region')

In [ ]:
score_name = 'Active_Immune_Engagement_Ratio'

numerator_cols = [
    'M1-like Active', 
    'DCs Active', 
    'T CD4 NK-like',
    'T (pan)'
]
# 'T (pan)'

denominator_cols = [
    'M2_TAM Active', 
    'M2_TAM Inactive', 
    'M2_TAM Suppressive', 
    'M1-like Inactive',
    'B Inactive'
]
#    'B Inactive'

# Calculate the Immune Engagement Ratio (IER)
# The + 1 in the denominator prevents division by zero (Laplace smoothing)
df_merged[score_name] = (
    df_merged[numerator_cols].sum(axis=1) / 
    (df_merged[denominator_cols].sum(axis=1) + 1)
)

if not score_name in final_df.columns.values:
    final_df = final_df.merge(df_merged[['unique_region', score_name]], on='unique_region')

print(final_df)

In [ ]:
adata.obs[['TMA', 'Region', 'code', 'condition', 'condition_final']].drop_duplicates().to_excel("%s/Metadata.xlsx" % output_dir)

In [ ]:
final_df[final_df['code'] == "PD_02"]

### Load data from Segmentaion from DRAQ5 and Hoechat1

In [ ]:
import re

data_dir = "/Users/lyongheo/TOOLS/Claude/Segmentation/"

TMAs = ["TMA1", "TMA2"]

seg_df = None

for TMA in TMAs:
    t_df = pd.read_csv("%s/%s/segmentation_results.csv" %(data_dir, TMA))
    t_df['unique_region'] = ["%s_Reg%d" %(TMA, int(reg.split('_')[0].replace("reg", ""))) for reg in t_df['region']]
    t_df = t_df.drop("region", axis = 1)
    if seg_df is None:
        seg_df = t_df
    else :
        seg_df = pd.concat([seg_df, t_df])

seg_df = seg_df.merge(final_df[['unique_region', 'code']], on = "unique_region")

In [ ]:
seg_cnt_df = (
    seg_df.assign(source_channel=seg_df["source_channel"].astype(str))
          .groupby(["unique_region", "code", "source_channel"], observed=True)
          .size()
          .unstack("source_channel", fill_value=0)
          .reset_index()
          .rename_axis(columns=None)
          [["unique_region", "DRAQ5", "Hoechst18"]]
)

seg_cnt_df

In [ ]:
if not "DRAQ5" in final_df.columns.values:
    final_df = final_df.merge(seg_cnt_df, on='unique_region')

cnt_df = adata.obs['unique_region'].value_counts().rename_axis('unique_region').reset_index(name="n_cells")

if not "n_cells" in final_df.columns.values:
    final_df = final_df.merge(cnt_df, on = "unique_region")
    
print(final_df)

In [ ]:
final_df['seg_qc'] = final_df['n_cells'] / final_df[['DRAQ5', 'Hoechst18']].max(axis=1)

In [ ]:
final_df['seg_qc'].describe()


In [ ]:
final_df['ndi_eDNA'] = (final_df['DRAQ5'] - final_df['Hoechst18']) / (final_df['DRAQ5'] + final_df['Hoechst18'])

final_df['APERxVIER'] = np.log2(final_df['Antigen_Presentation_Cell_Engagement_Ratio'] + 1) * np.log2(final_df['Vascular_Immune_Exclusion_Ratio'] + 1)

print(final_df)

## CN Bar I-association

In [ ]:
data_dir = "~/CODEX_08262025/Analysis/CN_Corrected/"

dt_cnmap = pd.concat([pd.read_csv("%sCN_MAP_Prior.csv" % data_dir), 
                      pd.read_csv("%sCN_MAP_Infected.csv" % data_dir), 
                      pd.read_csv("%sCN_MAP_Uninfected.csv" % data_dir)], ignore_index=True )

dt_cnmap['unique_region'] = ["TMA%s_Reg%s" % (uniq_region.split("_")[0].replace("TMA", ""), uniq_region.split("_")[1].replace("Reg", ""))
            for uniq_region in dt_cnmap.unique_region]
np.unique(dt_cnmap['unique_region'] )

dt_cnmap['combination_num'] = [comb_num.lstrip("(").rstrip(")").replace(",", "") for comb_num in dt_cnmap['combination_num']]


In [ ]:
## Neutrophil Zone - 0
## Vascular Zone - 1
## Cytotoxic T-cell Zone - 2
## Inflammatory Zone - 3
## Follicular Zone - 4
## Immune Priming Zone - 5
## Fibrotic Zone - 6

i_association = ["0 1", "2 3 4 5", "1 3 5", "3 5", "1 2"]

In [ ]:
counts_df = pd.crosstab(dt_cnmap['unique_region'], dt_cnmap['combination_num'])
cnt_df = pd.DataFrame({ 
    'i_assoc_cnt': counts_df.loc[:, i_association].sum(axis = 1)})

cnt_df

In [ ]:
if not "i_assoc_cnt" in final_df.columns.values:
    final_df = final_df.merge(cnt_df, on='unique_region')

In [ ]:
df2 = pd.read_excel("~/CODEX_08262025/DATA/CODEX_per-patient_clinical_FULL.xlsx",
                    index_col=None,
                    sheet_name = "Sheet_4_analysis")

if not "culture_status" in final_df.columns.values:
    final_df = final_df.merge(df2, on='code')
    
final_df

In [ ]:
final_df.to_excel("%s/All_Scores.xlsx" % output_dir, index = False)


In [ ]:
final_df = pd.read_excel("%s/All_Scores.xlsx" % output_dir, index_col=None)
final_df

### compare between Uninfected, Dormant, and Infected using mannwhitneyu

In [ ]:
stats_rows = []

df = pd.DataFrame({'colnames': ['Vascular_Immune_Exclusion_Ratio', 
                  'Normalized_VIER', 
                  'Antigen_Presentation_Cell_Engagement_Ratio', 
                  'Immune_Engagement_Ratio', 
                  'Active_Immune_Engagement_Ratio',
                  'APERxVIER'],
     'titles': ['Vascular Immune Exclusion Ratio by Patient Condition', 
                'Vascular-Immune Exclusion Ratio (Normalized) by Patient Condition',
                'Antigen Presentation Cell Engagement Ratio by Patient Condition', 
                'Immune Engagement Ratio by Patient Condition', 
                'Active Immune Engagement Ratio by Patient Condition',
                'APERxVIER by Patient Condition'],
     'y_labels': ['Vascular Immune Exclusion Ratio (VIER)', 
                 'Normalized Vascular Immune Exclusion Ratio (Normalized VIER)',
                 'Antigen Presentation Cell Engagement Ratio (APER)', 
                 'Immune Engagement Ratio (IER)', 
                 'Active Immune Engagement Ratio (AIER)',
                 'APER x VIER'
                 ]})
df

for i in range(0, df.shape[0]):
    y_col = df['colnames'][i]
    plot_title = df['titles'][i]
    y_label = df['y_labels'][i]
    
    df_merged = final_df[['code', 'condition_final']]
    score = final_df[y_col]
    #eps = score[score > 0].min() / 2
    df_merged["log2_%s" % y_col] = np.log2(score + 1)

    df_merged = (
        df_merged.groupby(["code", "condition_final"], as_index=False, observed=True)
                .agg(SCORE=("log2_%s" % y_col, "mean"),
                     n_regions=("log2_%s" % y_col, "size"))
    )


    # 2. Set up the plotting style
    plt.figure(figsize=(6, 8))
    sns.set_theme(style="whitegrid")

    # Define an order so the plot is always consistent
    order = ['Uninfected', 'Dormant', 'Infected']
    # Define a custom color palette
    palette = {'Uninfected': '#619cff', 'Infected': '#f8766d', 'Dormant': '#b4c640'}

    # 3. Create the Base Boxplot
    ax = sns.boxplot(
        data=df_merged, 
        x='condition_final', 
        y="SCORE",
        order=order,
        palette=palette,
        showfliers=False, # Hide outliers since the stripplot will show them
        width=0.5,
        boxprops={'alpha': 0.7} # Make boxes slightly transparent
    )

    # 4. Overlay the individual patient data points
    sns.stripplot(
        data=df_merged, 
        x='condition_final', 
        y="SCORE",
        order=order,
        color='black',
        alpha=0.6,
        jitter=True, # Spreads dots out horizontally so they don't overlap
        size=6
    )

    # 5. Add Statistical Significance Brackets
    pairs = [
        ("Uninfected", "Infected"),
        ("Uninfected", "Dormant"),
        ("Infected", "Dormant")
    ]

    # Calculate raw p-values manually
    raw_pvalues = []
    for p1, p2 in pairs:
        g1 = df_merged[df_merged['condition_final'] == p1]["SCORE"]
        g2 = df_merged[df_merged['condition_final'] == p2]["SCORE"]
        _, p = mannwhitneyu(g1, g2)
        raw_pvalues.append(p)

    # Apply FDR (BH) correction
    p_adj = multipletests(raw_pvalues, method='fdr_bh')[1]

    ## For table
    raw_pt = (final_df.groupby(["code", "condition_final"], observed=True)[y_col]
                      .mean().reset_index(name="RAW"))
    for k, (p1, p2) in enumerate(pairs):
        g1 = df_merged.loc[df_merged['condition_final'] == p1, "SCORE"].to_numpy()
        g2 = df_merged.loc[df_merged['condition_final'] == p2, "SCORE"].to_numpy()
        r1 = raw_pt.loc[raw_pt['condition_final'] == p1, "RAW"]
        r2 = raw_pt.loc[raw_pt['condition_final'] == p2, "RAW"]
        u1, _ = mannwhitneyu(g1, g2)
        n1, n2 = len(g1), len(g2)
        q = p_adj[k]
        stats_rows.append({
            "ScoreName": y_col,
            "Group1": p1,
            "Group2": p2,
            "G1_mean": g1.mean(),
            "G2_mean": g2.mean(),
            "N_G1": n1,
            "N_G2": n2,
            "Mean_diff": g1.mean() - g2.mean(),
            "G1_median": np.median(g1),
            "G2_median": np.median(g2),
            "G1_IQR": np.subtract(*np.percentile(g1, [75, 25])),
            "G2_IQR": np.subtract(*np.percentile(g2, [75, 25])),
            "G1_raw_mean": r1.mean(),
            "G2_raw_mean": r2.mean(),
            "G1_raw_median": r1.median(),
            "G2_raw_median": r2.median(),
            "U_statistic": u1,
            "Rank_biserial": 2 * u1 / (n1 * n2) - 1,
            "HL_shift": np.median(np.subtract.outer(g1, g2)),
            "raw p-value": raw_pvalues[k],
            "FDR": q,
            "Significance": "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns",
        })
    
    
    # Format exactly how you want (Decimal, no scientific notation)
    formatted_annotations = []
    for p in p_adj:
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        formatted_annotations.append(f"p = {p:.6f} ({stars})")

    # Initialize Annotator
    annotator = Annotator(ax, pairs, data=df_merged, x='condition_final', y="SCORE", order=order)

    # 1. Set the custom strings
    annotator.set_custom_annotations(formatted_annotations)

    # 2. Configure the layout
    annotator.configure(loc='inside')

    # 3. THE FIX: Use .annotate() instead of .apply_and_annotate()
    # This tells the library "Just draw the labels I gave you, don't calculate anything."
    annotator.annotate()

    # 6. Final Polish
    plt.title(plot_title, fontsize=14, pad=15)
    plt.ylabel(y_label, fontsize=12)
    plt.xlabel('', fontsize=12)

    # Save as a high-resolution PNG (Best for PowerPoint / Word)
    plt.savefig('%s/MannW_FDR_%s_Boxplot.png' %(output_dir, y_col), dpi=300, bbox_inches='tight')

    # Show the plot
    plt.show()
    plt.close()
    
stats_df = pd.DataFrame(stats_rows)
stats_df.to_excel('%s/MannW_FDR_stats_table.xlsx' % output_dir, index=False)
stats_df

### compare between Not Exposed, and Exposed using mannwhitneyu

In [ ]:
final_df['condtion_subgroup'] = ["Not Exposed" if cond == "Uninfected" else "Exposed" for cond in final_df['condition_final']]

final_df

In [ ]:
stats_rows = []

df = pd.DataFrame({'colnames': ['Vascular_Immune_Exclusion_Ratio', 
                  'Normalized_VIER', 
                  'Antigen_Presentation_Cell_Engagement_Ratio', 
                  'Immune_Engagement_Ratio', 
                  'Active_Immune_Engagement_Ratio',
                  'APERxVIER'],
     'titles': ['Vascular Immune Exclusion Ratio by Patient Condition', 
                'Vascular-Immune Exclusion Ratio (Normalized) by Patient Condition',
                'Antigen Presentation Cell Engagement Ratio by Patient Condition', 
                'Immune Engagement Ratio by Patient Condition', 
                'Active Immune Engagement Ratio by Patient Condition',
                'APERxVIER by Patient Condition'],
     'y_labels': ['Vascular Immune Exclusion Ratio (VIER)', 
                 'Normalized Vascular Immune Exclusion Ratio (Normalized VIER)',
                 'Antigen Presentation Cell Engagement Ratio (APER)', 
                 'Immune Engagement Ratio (IER)', 
                 'Active Immune Engagement Ratio (AIER)',
                 'APER x VIER'
                 ]})
df

for i in range(0, df.shape[0]):
    y_col = df['colnames'][i]
    plot_title = df['titles'][i]
    y_label = df['y_labels'][i]
    
    df_merged = final_df[['code', 'condtion_subgroup']]
    score = final_df[y_col]
    #eps = score[score > 0].min() / 2
    df_merged["log2_%s" % y_col] = np.log2(score + 1)

    df_merged = (
        df_merged.groupby(["code", "condtion_subgroup"], as_index=False, observed=True)
                .agg(SCORE=("log2_%s" % y_col, "mean"),
                     n_regions=("log2_%s" % y_col, "size"))
    )


    # 2. Set up the plotting style
    plt.figure(figsize=(6, 8))
    sns.set_theme(style="whitegrid")

    # Define an order so the plot is always consistent
    order = ["Not Exposed", "Exposed"]
    # Define a custom color palette
    palette = {'Not Exposed': '#619cff', 'Exposed': '#f8766d'}

    # 3. Create the Base Boxplot
    ax = sns.boxplot(
        data=df_merged, 
        x='condtion_subgroup', 
        y="SCORE",
        order=order,
        palette=palette,
        showfliers=False, # Hide outliers since the stripplot will show them
        width=0.5,
        boxprops={'alpha': 0.7} # Make boxes slightly transparent
    )

    # 4. Overlay the individual patient data points
    sns.stripplot(
        data=df_merged, 
        x='condtion_subgroup', 
        y="SCORE",
        order=order,
        color='black',
        alpha=0.6,
        jitter=True, # Spreads dots out horizontally so they don't overlap
        size=6
    )

    # 5. Add Statistical Significance Brackets
    pairs = [
        ("Not Exposed", "Exposed")
    ]

    # Calculate raw p-values manually
    raw_pvalues = []
    for p1, p2 in pairs:
        g1 = df_merged[df_merged['condtion_subgroup'] == p1]["SCORE"]
        g2 = df_merged[df_merged['condtion_subgroup'] == p2]["SCORE"]
        _, p = mannwhitneyu(g1, g2)
        raw_pvalues.append(p)

    # Apply FDR (BH) correction
    p_adj = multipletests(raw_pvalues, method='fdr_bh')[1]

    ## For table
    raw_pt = (final_df.groupby(["code", "condtion_subgroup"], observed=True)[y_col]
                      .mean().reset_index(name="RAW"))
    for k, (p1, p2) in enumerate(pairs):
        g1 = df_merged.loc[df_merged['condtion_subgroup'] == p1, "SCORE"].to_numpy()
        g2 = df_merged.loc[df_merged['condtion_subgroup'] == p2, "SCORE"].to_numpy()
        r1 = raw_pt.loc[raw_pt['condtion_subgroup'] == p1, "RAW"]
        r2 = raw_pt.loc[raw_pt['condtion_subgroup'] == p2, "RAW"]
        u1, _ = mannwhitneyu(g1, g2)
        n1, n2 = len(g1), len(g2)
        q = p_adj[k]
        stats_rows.append({
            "ScoreName": y_col,
            "Group1": p1,
            "Group2": p2,
            "G1_mean": g1.mean(),
            "G2_mean": g2.mean(),
            "N_G1": n1,
            "N_G2": n2,
            "Mean_diff": g1.mean() - g2.mean(),
            "G1_median": np.median(g1),
            "G2_median": np.median(g2),
            "G1_IQR": np.subtract(*np.percentile(g1, [75, 25])),
            "G2_IQR": np.subtract(*np.percentile(g2, [75, 25])),
            "G1_raw_mean": r1.mean(),
            "G2_raw_mean": r2.mean(),
            "G1_raw_median": r1.median(),
            "G2_raw_median": r2.median(),
            "U_statistic": u1,
            "Rank_biserial": 2 * u1 / (n1 * n2) - 1,
            "HL_shift": np.median(np.subtract.outer(g1, g2)),
            "raw p-value": raw_pvalues[k],
            "FDR": q,
            "Significance": "***" if q < 0.001 else "**" if q < 0.01 else "*" if q < 0.05 else "ns",
        })
    
    
    # Format exactly how you want (Decimal, no scientific notation)
    formatted_annotations = []
    for p in p_adj:
        stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        formatted_annotations.append(f"p = {p:.6f} ({stars})")

    # Initialize Annotator
    annotator = Annotator(ax, pairs, data=df_merged, x='condtion_subgroup', y="SCORE", order=order)

    # 1. Set the custom strings
    annotator.set_custom_annotations(formatted_annotations)

    # 2. Configure the layout
    annotator.configure(loc='inside')

    # 3. THE FIX: Use .annotate() instead of .apply_and_annotate()
    # This tells the library "Just draw the labels I gave you, don't calculate anything."
    annotator.annotate()

    # 6. Final Polish
    plt.title(plot_title, fontsize=14, pad=15)
    plt.ylabel(y_label, fontsize=12)
    plt.xlabel('', fontsize=12)

    # Save as a high-resolution PNG (Best for PowerPoint / Word)
    plt.savefig('%s/Subgroup_MannW_FDR_%s_Boxplot.png' %(output_dir, y_col), dpi=300, bbox_inches='tight')

    # Show the plot
    plt.show()
    plt.close()
    
stats_df = pd.DataFrame(stats_rows)
stats_df.to_excel('%s/Subgroup_MannW_FDR_stats_table.xlsx' % output_dir, index=False)
stats_df

In [ ]:
# Exact permutation is used only when n <= EXACT_MAX_N and there are no ties.
# n = 9 -> 362,880 permutations, which is the practical ceiling.
EXACT_MAX_N = 9

# --------------------------------------------------------------------------- #
# Spearman engine
# --------------------------------------------------------------------------- #
def _rho_no_ties(rx, ry, n):
    """Spearman rho from tie-free rank vectors."""
    d = rx - ry
    return 1.0 - 6.0 * np.sum(d * d) / (n * (n * n - 1.0))

def _exact_perm_p(x, y):
    """Two-sided exact p over all n! orderings (no ties assumed)."""
    n = len(x)
    rx = rankdata(x)
    ry = rankdata(y)
    rho_obs = _rho_no_ties(rx, ry, n)
    hits = 0
    for perm in permutations(ry):
        if abs(_rho_no_ties(rx, np.asarray(perm), n)) >= abs(rho_obs) - 1e-12:
            hits += 1
    return rho_obs, hits / factorial(n)
 
def sp_row(x, y, cohort, metric, target):
    """One correlation row. Mirrors sp_row() in the R script."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    n = len(x)
 
    row = {"cohort": cohort, "spatial_metric": metric, "target": target,
           "n": n, "rho": np.nan, "p": np.nan, "method": "not run"}
 
    if n < 3:
        row["method"] = "n<3"
        return row
 
    ties = (len(np.unique(x)) < n) or (len(np.unique(y)) < n)
 
    try:
        if not ties and n <= EXACT_MAX_N:
            rho, p = _exact_perm_p(x, y)
            method = "spearman-exact-perm"
        else:
            res = spearmanr(x, y)
            rho, p = float(res.statistic), float(res.pvalue)
            method = "spearman-approx(ties)" if ties else "spearman-approx"
    except Exception as err:                      # noqa: BLE001
        row["method"] = f"spearmanr error: {err}"
        return row
 
    row.update(rho=rho, p=p, method=method)
    return row
 
# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #
def run_item1(df, out_dir="item1_outputs", rank_desc=True):
    """
    Parameters
    ----------
    df : pd.DataFrame
        Region-level frame (final_df).
    out_dir : str
        Directory for the CSV / legend outputs.
    rank_desc : bool
        True -> rank 1 = most severe / most eDNA.
 
    Returns
    -------
    dict with keys 'patient', 'summary', 'ranks'.
    """
    os.makedirs(out_dir, exist_ok=True)
    df = df.copy()
 
    # ---- per-region derived metrics --------------------------------------- #
    df["VIER"] = np.log2(df["Vascular_Immune_Exclusion_Ratio"] + 1)
    df["Normalized_VIER_log2"] = np.log2(df["Normalized_VIER"] + 1)
    df["i_density"] = df["i_assoc_cnt"] / df["n_cells"]
    df["eDNA_use"] = df["ndi_eDNA"]                       # already computed
    df["Days_r"] = df["Days_Surgery_to_Other"].replace(-9, np.nan)
    # APERxVIER is taken as-is (already built from log2 scores)
 
    def first_val(z):
        z = z.dropna()
        return z.iloc[0] if len(z) else np.nan
 
    # ---- collapse regions -> patient -------------------------------------- #
    pt = (df.groupby("code", observed=True)
            .agg(condition=("condition_final", "first"),
                 VIER=("VIER", "mean"),
                 Normalized_VIER=("Normalized_VIER_log2", "mean"),
                 APERxVIER=("APERxVIER", "mean"),
                 i_density=("i_density", "mean"),
                 eDNA=("eDNA_use", "mean"),
                 culture=("culture_status", first_val),
                 Days=("Days_r", first_val),
                 n_regions=("unique_region", "size"))
            .reset_index())
 
    # Days is dormant-only: null it for every non-dormant patient
    pt.loc[pt["condition"] != "Dormant", "Days"] = np.nan
 
    for col in ["VIER", "Normalized_VIER", "APERxVIER", "i_density",
                "eDNA", "culture", "Days", "n_regions"]:
        pt[col] = pd.to_numeric(pt[col], errors="coerce")
 
    n_pt = len(pt)
    dorm = pt[pt["condition"] == "Dormant"]
    n_dorm = len(dorm)
    n_dorm_days = int(dorm["Days"].notna().sum())
    print(f"[info] {n_pt} patients across {len(df)} regions; dormant n = {n_dorm}")
 
    # sanity: culture_status / Days should be constant within a patient
    incon = (df.groupby("code", observed=True)[["culture_status", "Days_r"]]
               .nunique(dropna=True).gt(1).any(axis=1))
    if incon.any():
        print("[warn] culture_status or Days varies within these patients: "
              f"{list(incon[incon].index)}")
 
    # ---- correlations ------------------------------------------------------ #
    metrics = ["VIER", "Normalized_VIER", "APERxVIER", "i_density"]
    fc = f"full_n{n_pt}"
    dc = f"dormant_n{n_dorm}"
 
    rows = []
    for m in metrics:                                   # A: full cohort
        rows.append(sp_row(pt[m], pt["eDNA"], fc, m, "eDNA"))
        rows.append(sp_row(pt[m], pt["culture"], fc, m, "culture_status"))
    for m in metrics:                                   # B: dormant subgroup
        rows.append(sp_row(dorm[m], dorm["Days"], dc, m, "Days_Surgery_to_Other"))
        rows.append(sp_row(dorm[m], dorm["eDNA"], dc, m, "eDNA"))
 
    summary_tbl = pd.DataFrame(rows)
 
    # ---- BH-FDR within each cohort x target family ------------------------- #
    summary_tbl["FDR"] = np.nan
    summary_tbl["fdr_family"] = summary_tbl["cohort"] + " | " + summary_tbl["target"]
    summary_tbl["fdr_family_size"] = np.nan
    for fam, idx in summary_tbl.groupby("fdr_family", observed=True).groups.items():
        sub = summary_tbl.loc[idx]
        keep = sub.index[sub["p"].notna()]
        summary_tbl.loc[idx, "fdr_family_size"] = len(keep)
        if len(keep):
            summary_tbl.loc[keep, "FDR"] = multipletests(
                summary_tbl.loc[keep, "p"], method="fdr_bh")[1]
 
    summary_tbl["Significance"] = pd.cut(
        summary_tbl["FDR"], bins=[-np.inf, 0.001, 0.01, 0.05, np.inf],
        labels=["***", "**", "*", "ns"]).astype(object)
    summary_tbl.loc[summary_tbl["FDR"].isna(), "Significance"] = np.nan
 
    summary_tbl = summary_tbl[["cohort", "spatial_metric", "target", "n", "rho",
                               "p", "FDR", "Significance", "method",
                               "fdr_family", "fdr_family_size"]]
    summary_tbl.to_csv(os.path.join(out_dir, "correlation_summary.csv"), index=False)
 
    # ---- per-patient rank tables (vs eDNA, full cohort) -------------------- #
    asc = not rank_desc
    ranks = {}
    for m in metrics:
        tt = pd.DataFrame({
            "code": pt["code"],
            "condition": pt["condition"],
            "spatial_metric": m,
            "spatial_score": pt[m],
            "spatial_rank": pt[m].rank(ascending=asc, method="average"),
            "eDNA": pt["eDNA"],
            "eDNA_rank": pt["eDNA"].rank(ascending=asc, method="average"),
        }).sort_values("spatial_rank")
        tt.to_csv(os.path.join(out_dir, f"patient_ranks_{m}.csv"), index=False)
        ranks[m] = tt
 
    pt.to_csv(os.path.join(out_dir, "patient_level_table.csv"), index=False)
 
    # ---- legend ------------------------------------------------------------ #
    legend = [
        "LEGEND / METHODS NOTE (item #1)",
        f"Cohort: {n_pt} patients across {len(df)} regions "
        f"(max {int(pt['n_regions'].max())}/patient). "
        f"Dormant subgroup n={n_dorm} ({int(dorm['n_regions'].sum())} regions); "
        f"{n_dorm_days} with a usable Days value.",
        "Unit of analysis: PATIENT LEVEL (regions collapsed; multi-region patients",
        "  do not inflate n).",
        "Transformation: log2(x + 1) per region on VIER and Normalized_VIER only.",
        "  i_density (proportion), ndi_eDNA (bounded -1..1) and APERxVIER (already",
        "  built from log2 scores) are used untransformed.",
        "Aggregation: mean per patient for every metric. No seg_qc filtering.",
        "eDNA = ndi_eDNA = (#DRAQ5 - #Hoechst18) / (#DRAQ5 + #Hoechst18); the total",
        "  in the denominator removes the region-size effect.",
        "I-type association density = i_assoc_cnt / n_cells per region.",
        f"Full cohort (n={n_pt}): each spatial metric vs eDNA and culture_status.",
        f"Dormant subgroup (n={n_dorm}): each spatial metric vs Days_Surgery_to_Other",
        "  (condition_final=='Dormant'; -9 recoded to NA) and vs eDNA (underpowered).",
        f"Spearman rho via scipy. Exact permutation p over all n! orderings when",
        f"  n <= {EXACT_MAX_N} and no ties; tie-corrected asymptotic otherwise",
        "  (culture_status is binary -> always the approximation).",
        f"Subgroup exact two-sided floor at n={n_dorm_days}: 2/{factorial(n_dorm_days)}"
        f" = {2 / factorial(n_dorm_days):.4f} (at |rho|=1).",
        "FDR: Benjamini-Hochberg within each cohort x target family "
        "(4 metrics per family).",
        "  VIER and Normalized_VIER are the same measurement on two scales, so the",
        "  correction is slightly conservative. NB: after log2(x + 1) their patient",
        "  means and ranks are NOT identical, because the +1 is a different relative",
        "  amount on each scale.",
        f"Caveat: n={n_dorm} is ILLUSTRATIVE - report rho with the caveat, not",
        "  dressed up.",
        "  eDNA origin not established bacterial: internal consistency, NOT live",
        "  infection.",
    ]
    with open(os.path.join(out_dir, "LEGEND.txt"), "w") as fh:
        fh.write("\n".join(legend) + "\n")
    print("\n".join(legend))
    print("\nWrote outputs to:", os.path.abspath(out_dir))
 
    return {"patient": pt, "summary": summary_tbl, "ranks": ranks}

In [ ]:
res = run_item1(final_df, out_dir=output_dir)
res["summary"]